In [1]:
import pandas as pd
import json
import numpy as np
from sklearn.preprocessing import LabelEncoder

In [2]:
df_ban_heroes = pd.read_csv('../data/ban_heroes.csv')
df_pick_heroes = pd.read_csv('../data/pick_heroes.csv')

df_draft_heroes = df_ban_heroes.merge(df_pick_heroes, on='game_id')
df_draft_heroes = df_draft_heroes.rename(columns={
  'team_1_ban_1': 'step_1', 'team_2_ban_1': 'step_2', 'team_1_ban_2': 'step_3', 'team_2_ban_2': 'step_4',
  'team_1_ban_3': 'step_5', 'team_2_ban_3': 'step_6', 'team_1_pick_1': 'step_7', 'team_2_pick_1': 'step_8',
  'team_2_pick_2': 'step_9', 'team_1_pick_2': 'step_10', 'team_1_pick_3': 'step_11', 'team_2_pick_3': 'step_12',
  'team_2_ban_4': 'step_13', 'team_1_ban_4': 'step_14', 'team_2_ban_5': 'step_15', 'team_1_ban_5': 'step_16',
  'team_2_pick_4': 'step_17', 'team_1_pick_4': 'step_18', 'team_1_pick_5': 'step_19', 'team_2_pick_5': 'step_20'
})
df_draft_heroes = df_draft_heroes[['game_id', 'step_1', 'step_2', 'step_3', 'step_4', 'step_5', 'step_6', 'step_7', 'step_8', 'step_9', 'step_10',
                                   'step_11', 'step_12', 'step_13', 'step_14','step_15', 'step_16', 'step_17', 'step_18', 'step_19', 'step_20']]
df_draft_heroes.head()

,game_id,step_1,step_2,step_3,step_4,step_5,step_6,step_7,step_8,step_9,...,step_11,step_12,step_13,step_14,step_15,step_16,step_17,step_18,step_19,step_20
0,1,fanny,chip,zhask,ling,hayabusa,zhuxin,luo yi,valentina,roger,...,moskov,arlott,x.borg,joy,hylos,nolan,chou,julian,minotaur,harith
1,2,zhuxin,chip,khufra,ling,moskov,fanny,valentina,roger,edith,...,harith,arlott,terizla,hayabusa,hylos,luo yi,vexana,minotaur,thamuz,joy
2,3,terizla,chip,zhuxin,zhask,harith,ling,valentina,khufra,moskov,...,claude,vexana,minotaur,hayabusa,carmilla,alpha,ruby,julian,hylos,nolan
3,4,terizla,chip,zhuxin,ling,harith,edith,valentina,roger,zhask,...,moskov,hylos,fanny,alpha,joy,hayabusa,guinevere,julian,x.borg,claude
4,5,zhuxin,chip,terizla,ling,zhask,edith,valentina,roger,harith,...,moskov,x.borg,fanny,luo yi,hayabusa,minotaur,guinevere,julian,paquito,novaria


### One Hot Encoding Draft Steps

In [17]:
with open('../data/heroes2id.json', 'r') as f:
  heroes2id = json.load(f)

total_heroes = len(heroes2id)
game_id = []
one_hot_heroes = []
labels = []

for _, row in df_draft_heroes.iterrows():
  heroes = np.zeros(total_heroes, dtype=np.int8)
  for i in range(1, 7):
    hero_name = row[f'step_{i}']
    hero_index = heroes2id[hero_name]
    heroes[hero_index] = 1
  label = row['step_7']
  game_id.append(row['game_id'])
  one_hot_heroes.append(list(heroes))
  labels.append(heroes2id[label])

In [18]:
df_encoded_heroes = pd.DataFrame({
  'game_id': game_id,
  'one_hot_heroes': one_hot_heroes,
  'label': labels
})
df_encoded_heroes.head()

,game_id,one_hot_heroes,label
0,1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",95
1,2,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",109
2,3,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",109
3,4,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",109
4,5,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",109


### Encoding Game Information

In [7]:
df_game_information = pd.read_csv('../data/game_information.csv')
df_game_information.head()

,game_id,date,team_1,team_2,game_number,team_winner,patch_version
0,1,9-Aug-2024,tlid,fnoc,1,fnoc,1.9.06
1,2,9-Aug-2024,fnoc,tlid,2,fnoc,1.9.06
2,3,9-Aug-2024,dewa,evos,1,dewa,1.9.06
3,4,9-Aug-2024,dewa,evos,2,evos,1.9.06
4,5,9-Aug-2024,dewa,evos,3,dewa,1.9.06


In [8]:
le = LabelEncoder()
df_game_information['patch_version_encoded'] = le.fit_transform(df_game_information['patch_version'])
df_game_information['team_1_encoded'] = le.fit_transform(df_game_information['team_1'])
df_game_information['team_2_encoded'] = le.fit_transform(df_game_information['team_2'])

df_game_information.head()


,game_id,date,team_1,team_2,game_number,team_winner,patch_version,patch_version_encoded,team_1_encoded,team_2_encoded
0,1,9-Aug-2024,tlid,fnoc,1,fnoc,1.9.06,0,8,4
1,2,9-Aug-2024,fnoc,tlid,2,fnoc,1.9.06,0,4,8
2,3,9-Aug-2024,dewa,evos,1,dewa,1.9.06,0,2,3
3,4,9-Aug-2024,dewa,evos,2,evos,1.9.06,0,2,3
4,5,9-Aug-2024,dewa,evos,3,dewa,1.9.06,0,2,3


In [9]:
df_game_information_encoded = df_game_information[['game_id', 'patch_version_encoded', 'team_1_encoded', 'team_2_encoded']]
df_game_information_encoded.head()

,game_id,patch_version_encoded,team_1_encoded,team_2_encoded
0,1,0,8,4
1,2,0,4,8
2,3,0,2,3
3,4,0,2,3
4,5,0,2,3


In [19]:
df = df_game_information_encoded.merge(df_encoded_heroes, on='game_id')
df

,game_id,patch_version_encoded,team_1_encoded,team_2_encoded,one_hot_heroes,label
0,1,0,8,4,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",95
1,2,0,4,8,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",109
2,3,0,2,3,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",109
3,4,0,2,3,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",109
4,5,0,2,3,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",109
...,...,...,...,...,...,...
206,207,1,8,7,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",123
207,208,1,8,7,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",125
208,209,1,7,8,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",105
209,210,1,8,7,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, ...",105


In [20]:
data = df.drop('game_id', axis=1).to_numpy()
X = []
y = []
for i in data:
  X.append(list(i[:3]) + i[3])
  y.append(i[-1])

X = np.array(X)
y = np.array(y)

print(X.shape)
print(y.shape)

(211, 131)
(211,)


In [21]:
np.savetxt('../data/processed/X_7steps_encoded.csv', X, delimiter=',', fmt='%d')
np.savetxt('../data/processed/y_7steps_encoded.csv', y, delimiter=',', fmt='%d')

In [ ]:
df_7steps = df_draft_heroes[['game_id', 'step_1', 'step_2', 'step_3', 'step_4', 'step_5', 'step_6', 'step_7']]

,game_id,step_1,step_2,step_3,step_4,step_5,step_6,step_7
0,1,fanny,chip,zhask,ling,hayabusa,zhuxin,luo yi
1,2,zhuxin,chip,khufra,ling,moskov,fanny,valentina
2,3,terizla,chip,zhuxin,zhask,harith,ling,valentina
3,4,terizla,chip,zhuxin,ling,harith,edith,valentina
4,5,zhuxin,chip,terizla,ling,zhask,edith,valentina
...,...,...,...,...,...,...,...,...
206,207,edith,fanny,phoveus,suyou,mathilda,chou,chip
207,208,phoveus,fanny,alpha,chou,chip,yve,suyou
208,209,fanny,chip,chou,suyou,hilda,mathilda,phoveus
209,210,alpha,fanny,chip,hilda,bane,suyou,phoveus


In [12]:
df_7steps_feat = df_7steps.merge(df_game_information_encoded, on='game_id')
df_7steps_feat

,game_id,step_1,step_2,step_3,step_4,step_5,step_6,step_7,patch_version_encoded,team_1_encoded,team_2_encoded
0,1,fanny,chip,zhask,ling,hayabusa,zhuxin,luo yi,0,8,4
1,2,zhuxin,chip,khufra,ling,moskov,fanny,valentina,0,4,8
2,3,terizla,chip,zhuxin,zhask,harith,ling,valentina,0,2,3
3,4,terizla,chip,zhuxin,ling,harith,edith,valentina,0,2,3
4,5,zhuxin,chip,terizla,ling,zhask,edith,valentina,0,2,3
...,...,...,...,...,...,...,...,...,...,...,...
206,207,edith,fanny,phoveus,suyou,mathilda,chou,chip,1,8,7
207,208,phoveus,fanny,alpha,chou,chip,yve,suyou,1,8,7
208,209,fanny,chip,chou,suyou,hilda,mathilda,phoveus,1,7,8
209,210,alpha,fanny,chip,hilda,bane,suyou,phoveus,1,8,7


In [14]:
X7 = []
y7 = []

for _, row in df_7steps_feat.iterrows():
  feat = []
  feat.append(row['patch_version_encoded'])
  feat.append(row['team_1_encoded'])
  feat.append(row['team_2_encoded'])
  for i in range(1, 7):
    hero = row[f'step_{i}']
    feat.append(heroes2id[hero])
  hero = row['step_7']
  label = heroes2id[hero]
  X7.append(feat)
  y7.append(label)

X7 = np.array(X7)
y7 = np.array(y7)

print(X7.shape)
print(y7.shape)


(211, 9)
(211,)


In [16]:
np.savetxt('../data/processed/X_7steps.csv', X7, delimiter=',', fmt='%d')
np.savetxt('../data/processed/y_7steps.csv', y7, delimiter=',', fmt='%d')